# Step 1: 用 `TianmoucDataReader` 认识 Tianmouc 数据

这一节直接展示 `TianmoucDataReader` 返回的 sample 字典，重点是看懂：

- COP：`F0`, `F1`, `F0_without_isp`, `F0_HDR`, ...
- AOP：`rawDiff`, `tsdiff`，以及可选的 `txydiff`
- 元信息：`meta`, `sysTimeStamp`, `dataRatio`

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / 'tianmoucv').exists() and (candidate / 'tianmoucv_example').exists():
        repo_root = candidate
        break
if repo_root is None:
    raise RuntimeError('找不到 tianmoucv 仓库根目录，请从仓库内打开本 Notebook。')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import matplotlib.pyplot as plt
import torch


## 1. 时间关系：一段 sample = 两帧 COP + 中间多帧 AOP

`N=1` 时，reader 返回从 `F0` 到 `F1` 的一个最小时间段。中间插入多帧 AOP：

```text
时间轴

COP:      F0 ------------------------------------------------------------- F1
          |                                                               |
AOP:      r0  r1  r2  r3  ...                         ...  r(T-2)  r(T-1)
          |   |   |   |                                      |       |
rawDiff:  [TD, SDL, SDR] for every AOP tick

sample['dataRatio'] ≈ AOP帧率 / COP帧率
rawDiff.shape       = [3, T, h, w]
                    = [TD/SDL/SDR, AOP时间索引, AOP高度, AOP宽度]
```

这也是 Tianmouc 的核心：低频彩色 COP 提供纹理与颜色，高频 AOP 提供时间差分和空间差分。

In [ ]:
from tianmoucv.data import TianmoucDataReader
from tianmoucv.utils import ensure_tutorial_dataset

info = ensure_tutorial_dataset()

# 创建 Tianmouc 数据读取器。N=1 表示一个 sample 含 F0、F1 两帧 COP，及其间的 AOP 序列。
dataset = TianmoucDataReader(str(info.sample_dir), N=1, print_info=True)

sample_idx = len(dataset) // 2
sample = dataset[sample_idx]
print('reader index =', sample_idx)
print('sample keys =')
for key in sorted(sample.keys()):
    value = sample[key]
    if hasattr(value, 'shape'):
        print(f'  {key}: shape={tuple(value.shape)}, dtype={value.dtype}')
    else:
        print(f'  {key}: {type(value).__name__}')

print('dataRatio =', sample['dataRatio'])
print('sysTimeStamp =', sample['sysTimeStamp'])
print('meta keys =', sorted(sample['meta'].keys()))

## 2. 空间尺寸关系：COP 是图像平面，AOP 是差分平面

常见输出尺寸关系如下。实际尺寸以你手头数据打印结果为准。

```text
COP / RGB 平面

F0, F1, F0_HDR:
┌──────────────────────────────────────────────┐
│                                              │  
│              RGB image [H_rgb, W_rgb, 3]     │ H_rgb
│                                              │
└──────────────────────────────────────────────┘
                 W_rgb

AOP / rawDiff 平面

rawDiff[0, t] = TD   ┌──────────────────────┐
rawDiff[1, t] = SDL  │                      │  
rawDiff[2, t] = SDR  │  AOP [h_aop, w_aop]  │ h_aop
                     │                      │
                     └──────────────────────┘
                              w_aop

reader 会额外给出：
- tsdiff: 把 TSD 上采样到 RGB 尺寸后的版本，常用于部分神经网络输入。
- txydiff: 当 reader 使用 `_correct_sd_upsampler=True` 时给出，把 SD 转成 Ix/Iy 后再对齐到 RGB 尺寸。
```

`rawDiff` 是最原始、最常用的 AOP 输入；很多 TianmouCV API 都从它开始。

In [ ]:
shape_keys = ['F0', 'F1', 'F0_without_isp', 'F0_HDR', 'rawDiff', 'tsdiff', 'txydiff']
for key in shape_keys:
    if key in sample:
        print(f'{key:15s}', tuple(sample[key].shape))
    else:
        print(f'{key:15s}', 'not in this sample')

## 3. COP 合成关系：RAW → RGB，RGB + SD → HDR

`TianmoucDataReader` 在读取 COP 时会执行默认 ISP，并基于同一时刻附近的 SD 做一个内置 HDR 融合结果：

```text
cone/*.tmdat
    │
    ├─ default_rgb_isp ───────────────► F0 / F1
    │                                      RGB，可直接 imshow
    │
    ├─ demosaic / no extra ISP ───────► F0_without_isp / F1_without_isp
    │                                      更接近原始输出
    │
rod/*.tmdat ─► rawDiff[t] ─► SD_t ─┐
                                   ├─ HDRRecon(SD_t, F_i) ─► F0_HDR / F1_HDR
F_i ───────────────────────────────┘
```

下面直接用 `matplotlib` 展示同一个 COP 时刻的三种结果。

In [ ]:
idx = 0
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(sample[f'F{idx}'].numpy())
plt.title(f'F{idx}: RGB after ISP')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(sample[f'F{idx}_without_isp'].numpy())
plt.title(f'F{idx}_without_isp')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(sample[f'F{idx}_HDR'].numpy())
plt.title(f'F{idx}_HDR')
plt.axis('off')

plt.tight_layout()
plt.show()

## 4. AOP 通道关系：TD + SDL + SDR

`rawDiff` 的三个通道固定为：

```text
rawDiff.shape = [3, T, H, W]

channel 0: TD   = Temporal Difference，时间差分
channel 1: SDL  = Spatial Difference Left，左向/斜向空间差分
channel 2: SDR  = Spatial Difference Right，右向/斜向空间差分

对某个 AOP 时间 t：

rawDiff[:, t]
┌─────────────┬─────────────┬─────────────┐
│ TD[t]       │ SDL[t]      │ SDR[t]      │
│ [H, W]      │ [H, W]      │ [H, W]      │
└─────────────┴─────────────┴─────────────┘
```

可视化差分信号时通常先乘一个 gain，再以正负差分颜色显示。这里显式调用 `tianmoucv.isp.vizDiff`，而不是教程封装。

In [ ]:
from tianmoucv.isp import vizDiff

def show_diff_channel(channel, gain=8):
    image = vizDiff(torch.as_tensor(channel), gain=gain, bg_color='black')/255.0
    if torch.is_tensor(image):
        image = image.detach().cpu().numpy()
    return image

mid_t = sample['rawDiff'].shape[1] // 2
frame = sample['rawDiff'][:, mid_t]
print('选择的 AOP 帧索引:', mid_t)
print('rawDiff[:, mid_t].shape =', tuple(frame.shape))

plt.figure(figsize=(12, 4))
for i, title in enumerate(['TD', 'SDL', 'SDR']):
    plt.subplot(1, 3, i + 1)
    plt.imshow(show_diff_channel(frame[i], gain=8))
    plt.title(title)
    plt.axis('off')
plt.tight_layout()
plt.show()

## 5. 从 SDL/SDR 到 Ix/Iy，再组成 `txydiff`

Tianmouc 原始 SD 是 `SDL/SDR` 两个方向的差分。很多图像处理算法更习惯使用水平/垂直梯度 `Ix/Iy`，所以 TianmouCV 提供 `SD2XY`：

```text
rawDiff[1:3, t] = [SDL, SDR]
              │
              ▼
          SD2XY(...)
              │
              ├── Ix[t]
              └── Iy[t]

如果 reader 使用 _correct_sd_upsampler=True：

rawDiff
  ├─ TD 上采样到 RGB 尺寸 ──────────────► txydiff[0]
  └─ SDL/SDR → SD2XY → Ix/Iy → 上采样 ─► txydiff[1], txydiff[2]

因此：
- tsdiff ≈ [TD, SDL, SDR] 的 RGB 尺寸版本
- txydiff ≈ [TD, Ix, Iy] 的 RGB 尺寸版本
```

下面先手动对一个 `rawDiff` 帧调用 `SD2XY`，再创建一个带 `_correct_sd_upsampler=True` 的 reader 查看 `txydiff`。

In [ ]:
from tianmoucv.isp import SD2XY

sd = sample['rawDiff'][1:, mid_t].permute(1, 2, 0) / 128.0
ix, iy = SD2XY(sd)
print('SD input shape:', tuple(sd.shape))
print('Ix shape:', tuple(ix.shape))
print('Iy shape:', tuple(iy.shape))

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(show_diff_channel(sample['rawDiff'][1, mid_t], gain=8))
plt.title('SDL')
plt.axis('off')

plt.subplot(1, 3, 2)
ix_blue_green = vizDiff(ix, gain=4, bg_color='black')/255.0
plt.imshow(ix_blue_green.numpy(), cmap='gray')
plt.title('Ix = SD2XY(...)[0]')
plt.axis('off')

plt.subplot(1, 3, 3)
iy_blue_green = vizDiff(iy, gain=4, bg_color='black')/255.0
plt.imshow(iy_blue_green.numpy(), cmap='gray')
plt.title('Iy = SD2XY(...)[1]')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
dataset_xy = TianmoucDataReader(
    str(info.sample_dir),
    N=1,
    print_info=False,
    _correct_sd_upsampler=True,
)
sample_xy = dataset_xy[sample_idx]
print('keys with _correct_sd_upsampler=True:', sorted(sample_xy.keys()))
print('txydiff shape:', tuple(sample_xy['txydiff'].shape))
print('txydiff channel meaning: [TD, Ix, Iy]')